# 02 Forecasting Models

This notebook trains and compares forecasting models for one selected city.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'src'))

import plotly.express as px

from data_processing import load_weather_data, clean_weather_data, get_temperature_column, prepare_city_timeseries
from modeling import get_model_features, train_forecasting_models

In [ ]:
raw_path = PROJECT_ROOT / 'data' / 'raw' / 'GlobalWeatherRepository.csv'
df = clean_weather_data(load_weather_data(raw_path))
temp_col = get_temperature_column(df)

city = 'New York'
if city not in df['location_name'].astype(str).unique():
    city = df['location_name'].astype(str).iloc[0]

city_df = prepare_city_timeseries(df, city, temp_col)
city_df.head()

In [ ]:
feature_cols = get_model_features(city_df, temp_col)
metrics, models, prediction_frame = train_forecasting_models(city_df, temp_col, feature_cols)
metrics

In [ ]:
plot_cols = ['last_updated', temp_col] + [c for c in ['Baseline', 'Random Forest', 'Gradient Boosting', 'Ensemble'] if c in prediction_frame.columns]
long_preds = prediction_frame[plot_cols].melt(id_vars='last_updated', var_name='Series', value_name='Temperature')
px.line(long_preds, x='last_updated', y='Temperature', color='Series', title=f'Actual vs Forecasted Temperature - {city}')